# Benchmark: Baseline vs Optimized Inference

Compares the **baseline** revision (`07a3478f`) with the **optimized** branch (`claude/optimize-inference-speed-lkCRI`).

Tests **prune-aware + retain_kv_cache** mode with both **Flash Attention 2** and **SDPA** backends.

Metrics: output token match, wall time per generation, tokens/sec.

## 1. Setup

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────
REPO_URL = "https://github.com/anujjamwal/cognitive-compression.git"  # ← update if private/different
BASELINE_REV = "07a3478f3873ec6d8f2abb38c5073abe15033181"
OPTIMIZED_BRANCH = "claude/optimize-inference-speed-lkCRI"
MODEL_ID = "anujjamwal/OpenMath-Nemotron-1.5B-PruneAware"
MAX_NEW_TOKENS = 512
NUM_SAMPLES = 5  # number of prompts to benchmark
BATCH_SIZE = 1   # keep 1 for clean wall-time comparison

In [ ]:
%%capture
!pip install transformers accelerate datasets trl math-verify sentencepiece flash-attn --no-build-isolation -q

In [ ]:
import os, sys, time, shutil, subprocess, gc, json
from pathlib import Path

import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"torch {torch.__version__}  |  CUDA {torch.version.cuda}  |  GPU: {torch.cuda.get_device_name(0)}")

## 2. Clone repo at both revisions

In [ ]:
BASE_DIR = Path("/content")
BASELINE_DIR = BASE_DIR / "cc_baseline"
OPTIMIZED_DIR = BASE_DIR / "cc_optimized"

def clone_at_rev(dest: Path, rev: str):
    """Clone the repo and check out a specific revision."""
    if dest.exists():
        shutil.rmtree(dest)
    subprocess.run(["git", "clone", REPO_URL, str(dest)], check=True)
    subprocess.run(["git", "checkout", rev], cwd=str(dest), check=True)
    print(f"  → {dest.name}: checked out {rev[:12]}")

print("Cloning baseline...")
clone_at_rev(BASELINE_DIR, BASELINE_REV)

print("Cloning optimized...")
clone_at_rev(OPTIMIZED_DIR, f"origin/{OPTIMIZED_BRANCH}")

## 3. Load model & tokenizer (shared)

In [ ]:
def load_model(attn_impl: str):
    """Load model with specified attention implementation."""
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=dtype, device_map="auto",
        attn_implementation=attn_impl,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    return model, tokenizer, dtype


def prepare_model(model, tokenizer, lib_dir: Path):
    """Add special tokens using prepare_base_model from the given revision."""
    sys.path.insert(0, str(lib_dir))
    # Force reimport in case of prior import from another revision
    for mod_name in list(sys.modules):
        if mod_name.startswith(("trainer", "custom_generate", "eval")):
            del sys.modules[mod_name]
    from trainer import prepare_base_model
    model, tokenizer = prepare_base_model(model, tokenizer)
    model.eval()
    return model, tokenizer

## 4. Build test prompts

In [ ]:
MATH_PROMPTS = [
    "Solve: What is the sum of all integer values of $n$ such that $\\frac{20}{2n-1}$ is an integer?",
    "If $f(x) = 3x^2 - 7$ and $g(x) = x + 1$, what is $f(g(2))$?",
    "Find the value of $x$ if $2^{x+1} = 32$.",
    "What is the remainder when $3^{100}$ is divided by 7?",
    "Simplify $\\sqrt{50} + \\sqrt{18}$.",
    "How many positive divisors does 360 have?",
    "If $\\log_2(x) + \\log_2(x-2) = 3$, find $x$.",
    "What is the coefficient of $x^3$ in the expansion of $(2x + 1)^5$?",
]

PROMPTS = MATH_PROMPTS[:NUM_SAMPLES]
print(f"Using {len(PROMPTS)} prompts for benchmark")

In [ ]:
def tokenize_prompt(tokenizer, prompt: str) -> dict:
    """Build chat-formatted input for a single prompt."""
    messages = [
        {"role": "system", "content": "You are a helpful math assistant. Show your reasoning step by step."},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(text, return_tensors="pt", padding=False)
    return {k: v.to("cuda") for k, v in enc.items()}

## 5. Benchmark helpers

In [ ]:
def load_generate_fn(lib_dir: Path):
    """Import _sample from the given revision's custom_generate."""
    sys.path.insert(0, str(lib_dir))
    for mod_name in list(sys.modules):
        if mod_name.startswith(("trainer", "custom_generate", "eval")):
            del sys.modules[mod_name]
    from custom_generate.generate import _sample
    return _sample


def run_benchmark(model, tokenizer, generate_fn, prompts, label: str):
    """Run generation on all prompts and collect results."""
    results = []
    
    for i, prompt in enumerate(prompts):
        inp = tokenize_prompt(tokenizer, prompt)
        input_len = inp["input_ids"].shape[1]
        
        # Warmup on first prompt
        if i == 0:
            with torch.inference_mode():
                _ = model.generate(
                    **inp,
                    max_new_tokens=16,
                    use_cache=True,
                    custom_generate=generate_fn,
                    processing_class=tokenizer,
                    prune_aware=True,
                )
            torch.cuda.synchronize()
        
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        
        with torch.inference_mode():
            out = model.generate(
                **inp,
                max_new_tokens=MAX_NEW_TOKENS,
                use_cache=True,
                custom_generate=generate_fn,
                processing_class=tokenizer,
                prune_aware=True,
            )
        
        torch.cuda.synchronize()
        wall = time.perf_counter() - t0
        
        sequences = out if isinstance(out, torch.Tensor) else out.sequences
        gen_tokens = sequences[0, input_len:]
        num_gen = gen_tokens.shape[0]
        decoded = tokenizer.decode(gen_tokens, skip_special_tokens=False)
        
        results.append({
            "prompt_idx": i,
            "label": label,
            "input_len": input_len,
            "gen_tokens": num_gen,
            "wall_sec": wall,
            "tok_per_sec": num_gen / wall if wall > 0 else 0,
            "token_ids": gen_tokens.cpu().tolist(),
            "decoded": decoded,
        })
        print(f"  [{label}] prompt {i}: {num_gen} tokens in {wall:.2f}s ({num_gen/wall:.1f} tok/s)")
    
    return results


def free_model():
    """Release GPU memory."""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

## 6. Run: Baseline (revision 07a3478f)

In [ ]:
print("=" * 60)
print("BASELINE — SDPA")
print("=" * 60)

model_sdpa, tok_sdpa, dtype = load_model("sdpa")
model_sdpa, tok_sdpa = prepare_model(model_sdpa, tok_sdpa, BASELINE_DIR / "lib")
gen_fn_baseline = load_generate_fn(BASELINE_DIR / "lib")

baseline_sdpa_results = run_benchmark(model_sdpa, tok_sdpa, gen_fn_baseline, PROMPTS, "baseline-sdpa")

del model_sdpa, tok_sdpa
free_model()

In [ ]:
print("=" * 60)
print("BASELINE — Flash Attention 2")
print("=" * 60)

try:
    model_fa2, tok_fa2, dtype = load_model("flash_attention_2")
    model_fa2, tok_fa2 = prepare_model(model_fa2, tok_fa2, BASELINE_DIR / "lib")
    gen_fn_baseline = load_generate_fn(BASELINE_DIR / "lib")
    
    baseline_fa2_results = run_benchmark(model_fa2, tok_fa2, gen_fn_baseline, PROMPTS, "baseline-fa2")
    
    del model_fa2, tok_fa2
    free_model()
except Exception as e:
    print(f"FA2 not available: {e}")
    baseline_fa2_results = None

## 7. Run: Optimized (current branch)

In [ ]:
print("=" * 60)
print("OPTIMIZED — SDPA")
print("=" * 60)

model_sdpa, tok_sdpa, dtype = load_model("sdpa")
model_sdpa, tok_sdpa = prepare_model(model_sdpa, tok_sdpa, OPTIMIZED_DIR / "lib")
gen_fn_optimized = load_generate_fn(OPTIMIZED_DIR / "lib")

optimized_sdpa_results = run_benchmark(model_sdpa, tok_sdpa, gen_fn_optimized, PROMPTS, "optimized-sdpa")

del model_sdpa, tok_sdpa
free_model()

In [ ]:
print("=" * 60)
print("OPTIMIZED — Flash Attention 2")
print("=" * 60)

try:
    model_fa2, tok_fa2, dtype = load_model("flash_attention_2")
    model_fa2, tok_fa2 = prepare_model(model_fa2, tok_fa2, OPTIMIZED_DIR / "lib")
    gen_fn_optimized = load_generate_fn(OPTIMIZED_DIR / "lib")
    
    optimized_fa2_results = run_benchmark(model_fa2, tok_fa2, gen_fn_optimized, PROMPTS, "optimized-fa2")
    
    del model_fa2, tok_fa2
    free_model()
except Exception as e:
    print(f"FA2 not available: {e}")
    optimized_fa2_results = None

## 8. Compare outputs

In [ ]:
def compare_outputs(results_a, results_b, label_a, label_b):
    """Compare generated token sequences between two runs."""
    if results_a is None or results_b is None:
        print(f"  Skipping comparison (one run is missing)")
        return
    
    print(f"\n{'─' * 60}")
    print(f"Output comparison: {label_a} vs {label_b}")
    print(f"{'─' * 60}")
    
    all_match = True
    for a, b in zip(results_a, results_b):
        idx = a["prompt_idx"]
        ids_a = a["token_ids"]
        ids_b = b["token_ids"]
        
        if ids_a == ids_b:
            print(f"  Prompt {idx}: EXACT MATCH ({len(ids_a)} tokens)")
        else:
            all_match = False
            # Find first divergence
            min_len = min(len(ids_a), len(ids_b))
            first_diff = min_len  # default: one is longer
            for k in range(min_len):
                if ids_a[k] != ids_b[k]:
                    first_diff = k
                    break
            print(f"  Prompt {idx}: DIVERGE at token {first_diff}  "
                  f"(len {len(ids_a)} vs {len(ids_b)})")
    
    if all_match:
        print(f"\n  ✓ All {len(results_a)} prompts produce identical output")
    else:
        print(f"\n  ⚠ Some outputs differ — review above")


# Same attention backend: baseline vs optimized (should be identical)
compare_outputs(baseline_sdpa_results, optimized_sdpa_results, "baseline-sdpa", "optimized-sdpa")
compare_outputs(baseline_fa2_results, optimized_fa2_results, "baseline-fa2", "optimized-fa2")

# Cross-backend: sdpa vs fa2 (may differ due to numerical precision)
compare_outputs(optimized_sdpa_results, optimized_fa2_results, "optimized-sdpa", "optimized-fa2")

## 9. Wall time comparison

In [ ]:
def summarize_timing(results, label):
    """Return summary stats for a set of benchmark results."""
    if results is None:
        return None
    walls = [r["wall_sec"] for r in results]
    tps = [r["tok_per_sec"] for r in results]
    gens = [r["gen_tokens"] for r in results]
    return {
        "label": label,
        "num_prompts": len(results),
        "total_tokens": sum(gens),
        "total_wall_sec": sum(walls),
        "mean_wall_sec": np.mean(walls),
        "median_wall_sec": np.median(walls),
        "mean_tok_per_sec": np.mean(tps),
        "median_tok_per_sec": np.median(tps),
    }


all_summaries = []
for results, label in [
    (baseline_sdpa_results, "baseline-sdpa"),
    (baseline_fa2_results, "baseline-fa2"),
    (optimized_sdpa_results, "optimized-sdpa"),
    (optimized_fa2_results, "optimized-fa2"),
]:
    s = summarize_timing(results, label)
    if s:
        all_summaries.append(s)

print(f"\n{'Label':<20} {'Prompts':>8} {'Tokens':>8} {'Total(s)':>10} {'Mean(s)':>10} {'Med(s)':>10} {'Mean tok/s':>12} {'Med tok/s':>12}")
print("─" * 100)
for s in all_summaries:
    print(f"{s['label']:<20} {s['num_prompts']:>8} {s['total_tokens']:>8} "
          f"{s['total_wall_sec']:>10.2f} {s['mean_wall_sec']:>10.2f} {s['median_wall_sec']:>10.2f} "
          f"{s['mean_tok_per_sec']:>12.1f} {s['median_tok_per_sec']:>12.1f}")

In [ ]:
# Speedup calculations
def calc_speedup(base_results, opt_results, label):
    if base_results is None or opt_results is None:
        return
    base_total = sum(r["wall_sec"] for r in base_results)
    opt_total = sum(r["wall_sec"] for r in opt_results)
    speedup = base_total / opt_total if opt_total > 0 else float("inf")
    print(f"{label}: {base_total:.2f}s → {opt_total:.2f}s  ({speedup:.2f}x speedup)")

print("\n=== Speedup Summary ===")
calc_speedup(baseline_sdpa_results, optimized_sdpa_results, "SDPA")
calc_speedup(baseline_fa2_results, optimized_fa2_results, "FA2")
calc_speedup(baseline_sdpa_results, optimized_fa2_results, "SDPA→FA2 (total)")

## 10. Per-prompt detail

In [ ]:
import pandas as pd

all_results = []
for res_list in [baseline_sdpa_results, baseline_fa2_results, optimized_sdpa_results, optimized_fa2_results]:
    if res_list:
        for r in res_list:
            all_results.append({
                "label": r["label"],
                "prompt": PROMPTS[r["prompt_idx"]][:60] + "...",
                "input_len": r["input_len"],
                "gen_tokens": r["gen_tokens"],
                "wall_sec": round(r["wall_sec"], 3),
                "tok_per_sec": round(r["tok_per_sec"], 1),
            })

df = pd.DataFrame(all_results)
df

## 11. Sample outputs (side by side)

In [ ]:
# Show first 500 chars of decoded output for each prompt, baseline vs optimized (SDPA)
for i in range(min(3, len(PROMPTS))):
    print(f"\n{'=' * 80}")
    print(f"Prompt {i}: {PROMPTS[i][:80]}")
    print(f"{'=' * 80}")
    
    base = baseline_sdpa_results[i]["decoded"][:500]
    opt = optimized_sdpa_results[i]["decoded"][:500]
    
    print(f"\n--- Baseline SDPA ---")
    print(base)
    print(f"\n--- Optimized SDPA ---")
    print(opt)
    
    if base == opt:
        print("\n  → IDENTICAL")
    else:
        print("\n  → DIFFERS")

## 12. Save results

In [ ]:
output = {
    "config": {
        "baseline_rev": BASELINE_REV,
        "optimized_branch": OPTIMIZED_BRANCH,
        "model_id": MODEL_ID,
        "max_new_tokens": MAX_NEW_TOKENS,
        "num_samples": NUM_SAMPLES,
        "gpu": torch.cuda.get_device_name(0),
        "torch_version": torch.__version__,
        "cuda_version": torch.version.cuda,
    },
    "summaries": all_summaries,
}

# Strip token_ids from saved results to keep file small
for label_key, res_list in [
    ("baseline_sdpa", baseline_sdpa_results),
    ("baseline_fa2", baseline_fa2_results),
    ("optimized_sdpa", optimized_sdpa_results),
    ("optimized_fa2", optimized_fa2_results),
]:
    if res_list:
        output[label_key] = [{k: v for k, v in r.items() if k != "token_ids"} for r in res_list]

with open("/content/benchmark_results.json", "w") as f:
    json.dump(output, f, indent=2, default=str)

print("Results saved to /content/benchmark_results.json")